In [4]:
import numpy as np
import pandas as pd

from md_Helpers import (
    ProjectPaths,
    SQLiteRunDatabase,
    ThermalizationConfig,
    run_thermalization,
)


# ============================================================
# Database
# ============================================================

paths = ProjectPaths()
database = SQLiteRunDatabase(paths.database)
database.initialize()


# ============================================================
# Scan configuration
# ============================================================

kT_values = np.round(np.arange(0.9, 1.001, 0.05), 8)

# Used only when no existing in-range anchor is found.
coarse_rho_guesses = [0.65, 0.63, 0.67]

pressure_min = -0.1
pressure_max = 0.2
density_step = 0.005

# Hard safety bounds so an unexpected EOS cannot create an endless scan.
minimum_density = 0.10
maximum_density = 1.00
maximum_steps_per_direction = 30

n_cells_values = [30, 35, 40, 45]

nsteps = 200_000
log_period = 1_000
dt = 0.002
seed = 1


# ============================================================
# Run or reuse one fixed-density EOS point
# ============================================================

def run_eos_point(n_cells, kT, rho):
    n_cells = int(n_cells)
    kT = round(float(kT), 8)
    rho = round(float(rho), 8)

    config = ThermalizationConfig(
        n_fcc_cells=n_cells,
        target_rho=rho,
        nsteps=nsteps,
        kT=kT,
        log_period=log_period,
        dt=dt,
        seed=seed,
        notes=(
            "Adaptive fixed-density EOS scan focused on "
            f"{pressure_min} <= Pressure_Mean <= {pressure_max}"
        ),
    )

    result = run_thermalization(
        config,
        project_paths=paths,
        database=database,
    )

    if result.get("status") != "Complete":
        raise RuntimeError(
            f"EOS point N_Cells={n_cells}, kT={kT}, rho={rho} "
            f"returned Status={result.get('status')!r}, "
            f"Run_ID={result.get('run_id')}."
        )

    row = database.get_thermalization(result["run_id"])
    if row is None:
        raise RuntimeError(
            f"Completed run {result['run_id']} has no Thermalization row"
        )

    row = dict(row)
    row["Requested_N_Cells"] = n_cells
    row["Requested_kT"] = kT
    row["Requested_Density"] = rho
    row["Action"] = (
        "reused" if result.get("skipped", False) else "created"
    )

    return row


# ============================================================
# Find an existing anchor for one temperature
# ============================================================

def existing_anchor(n_cells, kT):
    candidates = database.query_thermalizations(
        N_Cells=int(n_cells),
        Therm_kT=round(float(kT), 8),
        Therm_Seed=seed,
        dt=dt,
        Nsteps=nsteps,
        Ensemble="NVT",
        Phase_Separation_Status="Not_Separated",
        Pressure_Mean=(pressure_min, pressure_max),
    )

    if not candidates:
        return None

    target_pressure = 0.5 * (pressure_min + pressure_max)

    return min(
        candidates,
        key=lambda row: abs(
            float(row["Pressure_Mean"]) - target_pressure
        ),
    )


# ============================================================
# Adaptive EOS scan
# ============================================================

records = []

for n_cells in n_cells_values:
    n_cells = int(n_cells)

    print()
    print("=" * 70)
    print(
        f"N_Cells={n_cells}, "
        f"N_Particles={4 * n_cells**3:,}"
    )
    print("=" * 70)

    for kT in kT_values:
        kT = round(float(kT), 8)

        print()
        print(
            f"N_Cells={n_cells}, "
            f"Temperature kT={kT:.3f}"
        )

        # ----------------------------------------------------
        # 1. Look for an existing anchor
        # ----------------------------------------------------

        anchor = existing_anchor(n_cells, kT)

        if anchor is not None:
            anchor = dict(anchor)
            anchor["Requested_N_Cells"] = n_cells
            anchor["Requested_kT"] = kT
            anchor["Requested_Density"] = float(
                anchor["Density_End"]
            )
            anchor["Action"] = "existing_anchor"

            print(
                "  Existing anchor:"
                f" rho={anchor['Density_End']:.4f},"
                f" P={anchor['Pressure_Mean']:.6f},"
                f" Run_ID={anchor['Run_ID']}"
            )

        # ----------------------------------------------------
        # 2. If needed, try the coarse density guesses
        # ----------------------------------------------------

        if anchor is None:
            print(
                "  No existing in-range anchor; "
                "trying coarse densities"
            )

            for rho in coarse_rho_guesses:
                row = run_eos_point(n_cells, kT, rho)
                records.append(row)

                phase_status = row["Phase_Separation_Status"]
                pressure = float(row["Pressure_Mean"])

                print(
                    f"    rho={rho:.4f},"
                    f" P={pressure:.6f},"
                    f" phase={phase_status},"
                    f" action={row['Action']},"
                    f" Run_ID={row['Run_ID']}"
                )

                if (
                    phase_status == "Not_Separated"
                    and pressure_min <= pressure <= pressure_max
                ):
                    anchor = row
                    print("    Selected as anchor")
                    break

        if anchor is None:
            print(
                "  No homogeneous state landed in the target "
                "pressure interval; skipping this "
                "N_Cells/kT combination."
            )
            continue

        records.append(anchor)

        anchor_density = round(
            float(anchor["Density_End"]),
            8,
        )

        # ----------------------------------------------------
        # 3. Scan downward and upward from the anchor
        # ----------------------------------------------------

        for direction_name, direction in [
            ("down", -1),
            ("up", +1),
        ]:
            print(f"  Scanning density {direction_name}")

            for offset in range(
                1,
                maximum_steps_per_direction + 1,
            ):
                rho = round(
                    anchor_density
                    + direction * offset * density_step,
                    8,
                )

                if (
                    rho < minimum_density
                    or rho > maximum_density
                ):
                    print(
                        "    Stopping at density safety bound: "
                        f"rho={rho:.4f}"
                    )
                    break

                row = run_eos_point(
                    n_cells,
                    kT,
                    rho,
                )
                records.append(row)

                phase_status = row[
                    "Phase_Separation_Status"
                ]
                pressure = float(row["Pressure_Mean"])

                print(
                    f"    rho={rho:.4f},"
                    f" P={pressure:.6f},"
                    f" phase={phase_status},"
                    f" action={row['Action']},"
                    f" Run_ID={row['Run_ID']}"
                )

                if phase_status == "Separated":
                    print(
                        "    Stopping branch: "
                        "state phase separated"
                    )
                    break

                if (
                    direction < 0
                    and pressure < pressure_min
                ):
                    print(
                        "    Stopping downward branch: "
                        f"P < {pressure_min}"
                    )
                    break

                if (
                    direction > 0
                    and pressure > pressure_max
                ):
                    print(
                        "    Stopping upward branch: "
                        f"P > {pressure_max}"
                    )
                    break


# ============================================================
# Results
# ============================================================

if records:
    eos_scan = (
        pd.DataFrame(records)
        .drop_duplicates(subset=["Run_ID"])
        .sort_values(
            ["N_Cells", "Therm_kT", "Density_End"]
        )
        .reset_index(drop=True)
    )

    display(
        eos_scan[
            [
                "Run_ID",
                "Action",
                "N_Cells",
                "Therm_kT",
                "Density_End",
                "Pressure_Mean",
                "Pressure_SEM",
                "Phase_Separation_Status",
                "PE_Per_Particle_Mean",
            ]
        ]
    )
else:
    eos_scan = pd.DataFrame()
    print("No EOS states were found or created.")


N_Cells=30, N_Particles=108,000

N_Cells=30, Temperature kT=0.900
  Existing anchor: rho=0.7050, P=0.051947, Run_ID=20260918171659
  Scanning density down
    rho=0.7000, P=0.023484, phase=Not_Separated, action=reused, Run_ID=20260917191859
    rho=0.6950, P=-0.001095, phase=Not_Separated, action=reused, Run_ID=20260918171851
    rho=0.6900, P=-0.024281, phase=Not_Separated, action=reused, Run_ID=20260917191713
    rho=0.6850, P=-0.043874, phase=Not_Separated, action=reused, Run_ID=20260918172047
    rho=0.6800, P=-0.063776, phase=Not_Separated, action=reused, Run_ID=20260917191527
    rho=0.6750, P=-0.079908, phase=Not_Separated, action=reused, Run_ID=20260918172241
    rho=0.6700, P=-0.096460, phase=Not_Separated, action=reused, Run_ID=20260917191200
    rho=0.6650, P=-0.112048, phase=Not_Separated, action=reused, Run_ID=20260918172437
    Stopping downward branch: P < -0.1
  Scanning density up
    rho=0.7100, P=0.076419, phase=Not_Separated, action=reused, Run_ID=20260917192046
  

,Run_ID,Action,N_Cells,Therm_kT,Density_End,Pressure_Mean,Pressure_SEM,Phase_Separation_Status,PE_Per_Particle_Mean
0,20260918172437,reused,30,0.9,0.665,-0.112048,0.001004,Not_Separated,-4.234931
1,20260917191200,reused,30,0.9,0.670,-0.096460,0.000986,Not_Separated,-4.264934
2,20260918172241,reused,30,0.9,0.675,-0.079908,0.000852,Not_Separated,-4.294682
3,20260917191527,reused,30,0.9,0.680,-0.063776,0.000963,Not_Separated,-4.326041
4,20260918172047,reused,30,0.9,0.685,-0.043874,0.001069,Not_Separated,-4.356656
...,...,...,...,...,...,...,...,...,...
207,20260918231820,reused,45,1.0,0.655,0.130464,0.000570,Not_Separated,-4.106724
208,20260918015417,reused,45,1.0,0.660,0.150322,0.000543,Not_Separated,-4.136658
209,20260918232346,reused,45,1.0,0.665,0.172667,0.000602,Not_Separated,-4.166541
210,20260918015941,reused,45,1.0,0.670,0.193776,0.000648,Not_Separated,-4.196808


In [ ]:
eos_states = database.query_thermalizations(
    N_Cells=n_cells_values,
    Therm_kT=kT_values.tolist(),
    Therm_Seed=seed,
    dt=dt,
    Nsteps=nsteps,
    Ensemble="NVT",
    Phase_Separation_Status="Not_Separated",
    Pressure_Mean=(pressure_min, pressure_max),
)

eos_states = (
    pd.DataFrame(eos_states)
    .sort_values(["N_Cells", "Therm_kT", "Density_End"])
    .reset_index(drop=True)
)

display(eos_states)